# Workshop 3: what limits an RL system in practice

Workshops 1 and 2 were about algorithms. This one is about the two days (2026-09-19 to
2026-09-21) in which the algorithms stopped being the bottleneck and four other things
took over: **data**, **capacity**, **compute** and **measurement**. Every section rests on a
source read in-session (`learning/RESOURCES.md`) and on numbers from `experiments/`.

Same format: plain words, the real code or the real numbers, your turn with a `check_*` cell.
No peeking at `learning/workshop3_solutions.md` before a check fails twice. Reflections in the
last cell go to `learning/records/`. Run the setup cell first.

In [ ]:
import json, math, random
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

random.seed(0); torch.manual_seed(0)
RESULTS = []
def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok))); print(("✅ " if ok else "❌ ") + name + (f"  ({detail})" if detail else ""))
def close(a, b, tol=1e-3):
    a, b = torch.as_tensor(a, dtype=torch.float64), torch.as_tensor(b, dtype=torch.float64)
    return a.shape == b.shape and bool(torch.allclose(a, b, atol=tol, rtol=0))
def load_run(path):
    d = json.loads(Path(path).read_text()); return d["logs"], d["config"]
def series(logs, key):
    return [e["episode"] for e in logs if e.get(key) is not None], [e[key] for e in logs if e.get(key) is not None]
def games_m1(arm): return json.loads(Path(f"experiments/{arm}/games.json").read_text())["mate_in_one_rate"]
def audit(name, defender="self"): return json.loads(Path(f"experiments/blunder-audit/{name}.json").read_text())["defenders"][defender]
def wins(matrix_file):
    d = json.loads(Path(f"experiments/win-matrix/{matrix_file}").read_text()); out = {}
    for c in d["counts"]:
        cc = c["counts"]; w = cc.get("a_white_white_win", 0) + cc.get("a_black_black_win", 0); l = cc.get("a_white_black_win", 0) + cc.get("a_black_white_win", 0)
        out[(c["a"], c["b"])] = (w, l); out[(c["b"], c["a"])] = (l, w)
    return d, out
print("setup ok")

## 1. Punishment, two ways

**Plain words.** The blunder audit showed our self-play defender takes a hung piece about
half the time and delivers an allowed mate one time in eight, so in training a gift costs
half a piece and an allowed mate costs almost nothing. AlphaStar's league adds *exploiter*
agents "whose sole purpose is to expose weaknesses of the main agent" because "playing to
win is insufficient". We built a one-ply rules exploiter and pointed it two ways:

- **Learner-side (arm PUNISH):** on every board, with probability 0.5, the learner's own
  ply becomes the punishing move (mixture log-prob stored, as with label guidance).
- **Opponent-side (arm POOL_REF):** the five frozen pool opponents always play the
  punishing move; the learner's plies are untouched.

Both start from DEPTH20 and run 120 updates. Think before you look: in each arm, *who*
plays the punishing move, *whose* exploration changes, and what should that do to the
learner's own mate-in-one rate in its own games?

In [ ]:
# TASK: punish_arms
# For each arm fill: who plays the punishing move ("learner" / "opponent"), whether the learner's own
# exploration distribution changes (True / False), and the predicted direction of own-game mate-in-one
# relative to DEPTH20 ("down" / "flat").
MY_PUNISH = {"PUNISH":   {"who": ..., "explore_shift": ..., "own_m1": ...},
             "POOL_REF": {"who": ..., "explore_shift": ..., "own_m1": ...}}

In [ ]:
_base = games_m1("endgame-technique/DEPTH20"); _p = games_m1("punish-gifts/PUNISH"); _r = games_m1("punish-gifts/POOL_REF")
_truth = {"PUNISH": {"who": "learner", "explore_shift": True, "own_m1": "down" if _p < _base - 0.05 else "flat"},
          "POOL_REF": {"who": "opponent", "explore_shift": False, "own_m1": "down" if _r < _base - 0.05 else "flat"}}
check("punishment arms", MY_PUNISH == _truth, f"own-game m1: DEPTH20 {_base:.3f}, PUNISH {_p:.3f}, POOL_REF {_r:.3f}")
_a = {n: audit(n) for n in ("long3", "punish", "pool_ref")}
print("gifts per 100 plies:", {n: round(v["gift_material_per100"], 1) for n, v in _a.items()}, " own punish rate:", {n: round(v["punish_rate_material"], 2) for n, v in _a.items()})
print("lesson: the arm that punished most also lost its mates; a demonstrator that pays for captures shifts what the learner explores, exactly as shaping that pays for captures did.")

## 2. Held-out dials versus strength

**Plain words.** Balduzzi et al. 2018 ("Re-evaluating Evaluation"): win-rate tables and proxy
metrics are biased by *which* opponents and tasks you include; redundant same-lineage
agents inflate each other. Our five-way matrix of 2026-09-21 put the old lineage's best
checkpoint (LONG3, ~2,000 updates of curricula) against two fresh big-data priors and two
600-update runs from them. Below: the puzzle dials next to the decisive-game counts.

In [ ]:
_d, _w = wins("matrix_width_long.json")
_arms = {"LONG3": "endgame-technique/LONG3", "LONG128": "width/LONG128", "LONG192": "width/LONG192"}
_dials = {n: load_run(f"experiments/{p}/run.json")[0][-1] for n, p in _arms.items()}
print(f"{'arm':10s} {'m1':>6s} {'m2':>6s} {'m3':>6s} {'m4':>6s} {'end':>6s}   decisive W-L vs the other two")
for n in _arms:
    e = _dials[n]; others = [f"{o}: {_w[(n, o)][0]}-{_w[(n, o)][1]}" for o in _arms if o != n]
    print(f"{n:10s} {e['lichess_top1']:6.3f} {e['lichess_m2_top1']:6.3f} {e['lichess_m3_top1']:6.3f} {e['lichess_m4_top1']:6.3f} {e['lichess_end_top1']:6.3f}   {', '.join(others)}")
print("row means (score vs everyone incl. the two priors):", {a: round(sum(_d['table'][a].values()) / (len(_d['names']) - 1), 3) for a in _d["names"]})

In [ ]:
# TASK: dials_vs_strength
BEST_BY_PUZZLES = ...      # arm name with the highest mate-in-1 held-out top-1
BEST_BY_GAMES = ...        # arm name with the best row mean in the matrix
# One sentence: what does the puzzle set measure, and what do the games measure, such that these differ?
WHAT_EACH_MEASURES = """..."""

In [ ]:
_bp = max(_arms, key=lambda n: _dials[n]["lichess_top1"]); _bg = max(_d["names"], key=lambda a: sum(_d["table"][a].values()))
check("best by puzzles", BEST_BY_PUZZLES == _bp, f"engine {_bp}"); check("best by games", BEST_BY_GAMES == _bg, f"engine {_bg}")
print("your sentence:", WHAT_EACH_MEASURES)
print("expected direction: the puzzle sets measure the skills the curriculum taught (mates from given positions); games measure what the prior knew about ordinary positions. Read puzzles as skill dials, not as strength.")

## 3. Diagnosing a plateau: data or capacity?

**Plain words.** Five checkpoints sat at par with the small-data prior for a week. Two
hypotheses: the 128-filter network is full (capacity), or the prior it started from never
had enough human games (data). The supervised stage gives the cleanest test: if a wider
network trained on the *same* data lands at the same held-out accuracy, and training
accuracy already exceeds held-out, the data is the limit. AlphaGo's first stage used
strong-player games for the same reason: the prior can only be as good as the moves it
imitates.

Below: the last five log entries of the 128 and 192 pretraining runs on the 299k set.

In [ ]:
def pre_log(name): return [e for e in json.loads(Path(f"experiments/human-pretraining/{name}/pretrain_log.json").read_text()) if "train_top1" in e]
def final_top1(name): return next(e["final"]["eval_top1"] for e in json.loads(Path(f"experiments/human-pretraining/{name}/pretrain_log.json").read_text()) if "final" in e)
for n in ("sl", "sl192"):
    tail = pre_log(n)[-5:]
    print(n, "train top-1", [round(e["train_top1"], 3) for e in tail], "held-out", [round(e.get("eval_top1", float("nan")), 3) for e in tail], "final", round(final_top1(n), 3))

In [ ]:
# TASK: diagnose_plateau
def train_eval_gap(entries):
    """Mean (train_top1 - eval_top1) over entries that have both keys."""
    pairs = [(e["train_top1"], e["eval_top1"]) for e in entries if "eval_top1" in e]
    return ...

# If the network were capacity-limited, what should widening 128 -> 192 do to held-out top-1 on the same data?
IF_CAPACITY_LIMITED_192_WOULD = ...      # "rise" / "same"
# What did it do?  ("rise" / "same", judged at +-0.01)
OBSERVED_192 = ...
DIAGNOSIS = ...                          # "data" / "capacity"

In [ ]:
_gap = train_eval_gap(pre_log("sl")[-5:]); _ref = float(np.mean([e["train_top1"] - e["eval_top1"] for e in pre_log("sl")[-5:] if "eval_top1" in e]))
check("train_eval_gap", close([_gap], [_ref]), f"engine {_ref:.3f}")
_obs = "rise" if final_top1("sl192") - final_top1("sl") > 0.01 else "same"
check("capacity hypothesis prediction", IF_CAPACITY_LIMITED_192_WOULD == "rise")
check("observed", OBSERVED_192 == _obs, f"sl {final_top1('sl'):.3f} vs sl192 {final_top1('sl192'):.3f}")
check("diagnosis", DIAGNOSIS == ("data" if _obs == "same" and _ref > 0.03 else "capacity"))
print("confirmation: a 128-filter prior on 1.43M positions reached", round(final_top1("slbig128"), 3), "and beat LONG3 24-0 with zero RL.")

## 4. Width: nothing in SL, real in RL

**Plain words.** Ota et al. 2021: in deep RL, larger networks "do not lead to performance
improvement" by default, because training becomes unstable. Our two long runs from the
big-data priors (same recipe, 600 updates each) are the exception worth understanding:
the wider one won 28-16, kept par with its prior while the narrow one drifted below its
own, changed less per update (KL) and kept more moves alive (entropy). The plot hides the
two arms behind letters. Use those three signatures.

In [ ]:
_pair = {"LONG128": "experiments/width/LONG128/run.json", "LONG192": "experiments/width/LONG192/run.json"}
_rng = random.Random(5); _let = list("PQ"); _rng.shuffle(_let); HIDDEN4 = dict(zip(_let, _pair))
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for letter, arm in HIDDEN4.items():
    logs = load_run(_pair[arm])[0]
    for ax, key in zip(axes, ["lichess_top1", "prior_score", "mean_entropy_normalized_game", "mean_approx_kl"]):
        xs, ys = series(logs, key); ax.plot(xs, ys, marker="o" if key in ("lichess_top1", "prior_score") else None, ms=3, label=letter); ax.set_title(key)
for ax in axes: ax.legend(); ax.set_xlabel("update")
plt.tight_layout(); plt.show()

In [ ]:
# TASK: read_width
MY_WIDTH = {"P": ..., "Q": ...}           # "LONG128" or "LONG192"
# One sentence: which of the four panels convinced you, and why is that signature expected for the wider network?
WHICH_PANEL = """..."""

In [ ]:
check("width arms identified", MY_WIDTH == HIDDEN4, f"truth {HIDDEN4}")
print("your sentence:", WHICH_PANEL)
print("expected: the wider run keeps prior_score near 0.5 (the narrow one falls to 0.35), holds higher entropy and lower KL, and its mate-in-1 is still rising at 550. Decisive games: 28-16 for 192.")

## 5. Data scaling: how far does the prior curve go?

**Plain words.** Tuyls et al. 2023: imitation-learning loss and downstream return "scale
smoothly with the compute budget ... resulting in power laws". Our supervised priors trace
the same curve on a log axis: 299k -> 1.43M -> 2.85M positions at Elo >= 1500. A fourth prior
(high-Elo 256, Elo >= 2000) is evaluated on a *different* held-out set and does not belong
on the same axis; the win matrix is the only fair comparison across sets.

In [ ]:
_pts = []
for n, npos in (("sl", 298960), ("slbig128", 1425360), ("sl2m192", 2850720)):
    if Path(f"experiments/human-pretraining/{n}/pretrain_log.json").exists(): _pts.append((npos, final_top1(n)))
print("(positions, held-out top-1):", [(p, round(t, 3)) for p, t in _pts])
plt.figure(figsize=(6, 3.6)); plt.semilogx([p for p, _ in _pts], [t for _, t in _pts], marker="o"); plt.xlabel("training positions (log)"); plt.ylabel("held-out human top-1"); plt.grid(alpha=.3); plt.show()

In [ ]:
# TASK: scaling
def log_fit(points):
    """Least-squares fit top1 = a + b * log10(positions). Return (a, b)."""
    x = np.log10([p for p, _ in points]); y = np.array([t for _, t in points])
    b = ...
    a = ...
    return float(a), float(b)

# Extrapolate: what does the fit predict for 5.7M positions (two more months at 1500)?
MY_PREDICTION_5_7M = ...
# Why can the high-Elo prior's top-1 (evaluated on Elo>=2000 positions) NOT be placed on this curve? One sentence.
WHY_NOT_SAME_AXIS = """..."""

In [ ]:
_x = np.log10([p for p, _ in _pts]); _y = np.array([t for _, t in _pts]); _b, _a = np.polyfit(_x, _y, 1)
_mine = log_fit(_pts); check("log fit", close(_mine, [_a, _b], 1e-3), f"engine a={_a:.3f}, b={_b:.3f}")
_pred = _a + _b * math.log10(5.7e6); check("prediction at 5.7M", close([MY_PREDICTION_5_7M], [_pred], 0.005), f"fit says {_pred:.3f}")
print("your sentence:", WHY_NOT_SAME_AXIS)
print("expected: a different held-out set (strong players' moves are harder or easier to predict for reasons unrelated to data size); only games between the priors compare them.")

## 6. Measurement traps

**Plain words.** Henderson et al. 2018: "Without significance metrics and tighter
standardization of experimental reporting, it is difficult to determine whether improvements
... are meaningful." Ours, in miniature: LONG3 was reported with own-game mate-in-one 0.387.
The 40-game evaluation says 0.232. Both numbers are real; they are different measurements.
Find them.

In [ ]:
_l3 = load_run("experiments/endgame-technique/LONG3/run.json")[0][-1]
_g3 = json.loads(Path("experiments/endgame-technique/LONG3/games.json").read_text())
print("run.json last log keys containing 'top1':", sorted(k for k in _l3 if "top1" in k))
print("games.json keys:", sorted(_g3))

In [ ]:
# TASK: two_numbers
PROBE_KEY = ...          # the run.json key that gave 0.387 (the training-time self-play probe)
EVAL_KEY = ...           # the games.json key that gives 0.232 (40 sampled games, seed 0)
# With 40 games and ~70 mate-in-one chances, roughly how wide is the 95% band on a rate near 0.3? (0.05 / 0.1 / 0.2)
NOISE_BAND = ...

In [ ]:
check("probe key", PROBE_KEY == "selfplay_top1" and abs(_l3["selfplay_top1"] - 0.387) < 0.01, f"selfplay_top1 = {_l3['selfplay_top1']:.3f}")
check("eval key", EVAL_KEY == "mate_in_one_rate" and abs(_g3["mate_in_one_rate"] - 0.232) < 0.01, f"mate_in_one_rate = {_g3['mate_in_one_rate']:.3f} over {_g3['mate_in_one_opportunities']} chances")
_n = _g3["mate_in_one_opportunities"]; _band = 2 * 1.96 * math.sqrt(0.3 * 0.7 / _n)
check("noise band", NOISE_BAND == min((0.05, 0.1, 0.2), key=lambda b: abs(b - _band)), f"±1.96·sqrt(p(1-p)/n) gives a band of {_band:.2f}")

## 7. Compute: profile first, then believe the speedup

**Plain words.** A profile of three training updates on the CPU took 75 s: about 60 s in
network forward/backward (PPO + self-imitation), about 5 s in python-chess rules, the rest
in environment stepping. The Apple GPU was available all along; the command sandbox hid
it, and every run before 2026-09-21 ran on CPU. Measured outside the sandbox:

| | CPU | GPU (MPS) |
|---|---|---|
| forward, 256 positions, 128 filters | 310 ms | 11 ms |
| training step, 128 filters | 977 ms | 38 ms |
| training step, 192 filters | 1388 ms | 64 ms |

The owner kept the update counts unchanged so results stay comparable.

In [ ]:
# TASK: amdahl
def speedup_ceiling(total_s, network_s):
    """If the network part became free, by what factor would the run speed up?"""
    return ...
MY_CEILING = ...                    # for 75 s total, 60 s network
MY_STEP_SPEEDUP_128 = ...           # 977 / 38
# Why did the wall clock of RL192 fall from 45 min (CPU) to 9 min, i.e. only ~5x, when the GPU step is 25x faster? One sentence.
WHY_ONLY_5X = """..."""

In [ ]:
check("ceiling", close([speedup_ceiling(75, 60), MY_CEILING], [5.0, 5.0], 1e-6)); check("step speedup", close([MY_STEP_SPEEDUP_128], [977 / 38], 0.5))
print("your sentence:", WHY_ONLY_5X)
print("expected: Amdahl — the environment stepping and rules checks stay on the CPU, so once the network is nearly free they set the floor.")

## 8. Your words

Three to five sentences: which of the four limits (data, capacity, compute, measurement)
surprised you most, what you would check first the next time a run plateaus, and where
outside chess the same shape appears (pretraining data quality in language models,
simulator fidelity in robotics, proxy metrics in any dashboard).

In [ ]:
MY_REFLECTION = """
...
"""
print(MY_REFLECTION); print(f"checks passed: {sum(ok for _, ok in RESULTS)}/{len(RESULTS)}")